# LLM 응답 캐싱(Caching) 실습

같은 질문을 여러 번 물어볼 때마다 매번 LLM API를 호출하면 시간도 오래 걸리고 비용도 반복해서 든다. LangChain의 **캐시(cache)** 기능을 쓰면, 한 번 물어본 질문(정확히는 같은 프롬프트)에 대한 답을 저장해뒀다가, 똑같은 질문이 다시 들어오면 API를 다시 호출하지 않고 저장된 답을 즉시 돌려준다. `%%time` 매직 명령으로 각 호출에 실제로 걸린 시간을 측정해가며 캐시 적용 전/후 속도 차이를 직접 확인한다.

## 1. 환경변수 로드

`.env`에 저장된 `OPENAI_API_KEY`를 불러온다.

In [1]:
from dotenv import load_dotenv

# .env 파일의 OPENAI_API_KEY 등 환경변수를 불러온다.
load_dotenv()

True

## 2. LLM과 체인 준비

"{country}에 대해서 200자 내외로 요약해줘"라는 프롬프트에 나라 이름을 넣어 요약을 받는 간단한 체인을 만든다. 아직 캐시를 설정하지 않은 상태다.

In [2]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate

llm = ChatOpenAI(
    model="gpt-5.6-luna",
)

prompt = PromptTemplate.from_template("{country}에 대해서 200자 내외로 요약해줘")

chain = prompt | llm

## 3. 캐시 없이 호출 (기준 시간 측정)

`%%time`은 주피터 셀 매직 명령으로, 그 셀을 실행하는 데 걸린 시간을 셀 실행 직후에 출력해준다. 아직 캐시가 없으므로 실제로 LLM API를 호출해서 응답을 받아온다. (실행 결과: Wall time 약 2.93초)

In [3]:
%%time
# 아직 캐시가 없어서 실제로 API를 호출한다. 응답이 오기까지의 시간을 %%time으로 측정.
response = chain.invoke({"country": "한국"})
print(response.content)

대한민국은 동아시아 한반도 남부에 위치한 민주공화국으로, 수도는 서울이다. 빠른 산업화와 민주화를 바탕으로 세계적인 경제·기술 강국으로 성장했으며, 반도체·자동차·조선·문화콘텐츠 산업이 발달했다. 한글, K-pop, 드라마와 음식 등 풍부한 문화도 세계적으로 사랑받는다. 삼면이 바다로 둘러싸였고 사계절이 뚜렷하며, 역사와 전통이 현대적인 도시 환경과 조화를 이룬다.
CPU times: total: 312 ms
Wall time: 2.93 s


## 4. InMemoryCache 설정 후 첫 호출

`set_llm_cache(InMemoryCache())`로 캐시를 활성화한다. `InMemoryCache`는 프로그램이 실행되는 동안(메모리 상에)만 유지되는 캐시로, 이 코드를 실행하는 순간부터 새로 캐시가 시작된다. 캐시를 막 켠 직후라 아직 저장된 답이 없으므로, 이번 호출도 실제 API를 호출한다 (Wall time 약 3.23초로 여전히 느림). 다만 이 호출부터는 결과가 캐시에 저장된다.

In [4]:
%%time
from langchain_core.globals import set_llm_cache
from langchain_core.caches import InMemoryCache

# 캐시를 메모리에 저장하도록 설정. 이 시점 이후의 호출 결과부터 캐시된다.
set_llm_cache(InMemoryCache())

# 캐시가 방금 켜졌으므로 아직 저장된 답이 없다 -> 이번 호출은 실제 API를 호출한다.
response = chain.invoke({"country": "한국"})
print(response.content)

대한민국은 동아시아 한반도 남쪽에 위치한 민주공화국으로, 수도는 서울이다. 삼면이 바다로 둘러싸였으며 사계절이 뚜렷하고 산지가 많다. 한국전쟁 이후 빠른 경제성장과 산업화를 이루어 반도체, 자동차, 조선, 정보통신 분야에서 세계적인 경쟁력을 갖췄다. 한글, 전통문화, K-pop, 영화와 드라마 등 풍부한 문화유산과 현대 대중문화를 함께 발전시키고 있다. 그러나 저출생, 고령화, 수도권 집중, 남북 분단 등의 과제도 안고 있다.
CPU times: total: 15.6 ms
Wall time: 3.23 s


## 5. 완전히 같은 질문을 다시 호출 (캐시 적중)

바로 앞과 똑같은 입력(`{"country": "한국"}`)으로 다시 호출한다. 이번에는 캐시에 저장된 답이 있으므로 API를 호출하지 않고 캐시에서 바로 꺼내온다. 실행 결과를 보면 Wall time이 3.22 **밀리초**로, 앞의 3.23**초**보다 1000배 가까이 빨라진 것을 확인할 수 있다. 답변 내용도 캐시된 그대로라서 이전 호출과 완전히 동일하다.

In [5]:
%%time
# 바로 앞 호출과 완전히 같은 입력 -> 캐시에 저장된 답을 그대로 반환 (API 재호출 없음).
# 실행 시간이 초 단위 -> 밀리초 단위로 확 줄어드는 것을 확인할 수 있다.
response = chain.invoke({"country": "한국"})
print(response.content)

대한민국은 동아시아 한반도 남쪽에 위치한 민주공화국으로, 수도는 서울이다. 삼면이 바다로 둘러싸였으며 사계절이 뚜렷하고 산지가 많다. 한국전쟁 이후 빠른 경제성장과 산업화를 이루어 반도체, 자동차, 조선, 정보통신 분야에서 세계적인 경쟁력을 갖췄다. 한글, 전통문화, K-pop, 영화와 드라마 등 풍부한 문화유산과 현대 대중문화를 함께 발전시키고 있다. 그러나 저출생, 고령화, 수도권 집중, 남북 분단 등의 과제도 안고 있다.
CPU times: total: 0 ns
Wall time: 3.22 ms


## 6. SQLiteCache로 교체 (디스크에 영구 저장)

`InMemoryCache`는 프로그램(커널)을 껐다 켜면 사라진다. `SQLiteCache`는 지정한 파일(`cache/llm_cache.db`)에 캐시를 저장해서, 노트북을 재시작해도 캐시가 그대로 남아있다. `langchain_community`의 `SQLiteCache`는 실행 시 "sunset(지원 종료 예정)" 경고가 뜨는데, 이는 `langchain-community` 패키지 자체가 점차 별도의 통합 패키지들로 이전되고 있어서 나오는 경고이고, 기능 자체는 정상 동작한다.

In [6]:
from langchain_community.cache import SQLiteCache
from langchain_core.globals import set_llm_cache
import os

# cache 폴더가 없으면 새로 만든다.
if not os.path.exists("cache"):
    os.makedirs("cache")

# 캐시를 메모리가 아니라 파일(cache/llm_cache.db)에 저장하도록 설정.
# -> 노트북(커널)을 재시작해도 캐시가 유지된다.
set_llm_cache(SQLiteCache(database_path="cache/llm_cache.db"))

C:\Users\user\AppData\Local\Temp\ipykernel_1252\538405311.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.cache import SQLiteCache


## 7. SQLiteCache로 첫 호출 (역시 캐시 미스)

`InMemoryCache`와는 별개의 새 캐시 저장소이므로, 이 시점에는 아직 SQLite 캐시에 저장된 답이 없다. 그래서 이번에도 실제 API를 호출한다 (Wall time 약 2.86초).

In [7]:
%%time
# 방금 새로 설정한 SQLiteCache에는 아직 저장된 답이 없다 -> 실제 API 호출.
response = chain.invoke({"country":"한국"})
print(response.content)

대한민국은 동아시아 한반도 남부에 위치한 민주공화국으로, 수도는 서울이다. 오랜 역사와 전통문화를 바탕으로 한국전쟁 이후 빠른 산업화와 민주화를 이루었다. 세계적인 제조업·정보기술 강국이며, 반도체·자동차·조선·바이오 산업이 발달했다. 한류, 영화, 음악, 음식도 국제적으로 큰 영향력을 지닌다. 북한과 분단된 상황에서 안보와 평화, 저출생·고령화 같은 사회 문제에 대응하고 있다.
CPU times: total: 15.6 ms
Wall time: 2.86 s


## 8. 같은 질문을 다시 호출 (SQLite 캐시 적중)

바로 앞 호출로 SQLite 캐시 파일에 답이 저장되었으므로, 다시 같은 질문을 호출하면 파일에서 바로 읽어온다. Wall time이 다시 밀리초 단위(6.11ms)로 크게 줄어드는 것을 확인할 수 있다. `InMemoryCache`와 다른 점은, 이 캐시는 `cache/llm_cache.db` 파일에 저장되어 있어서 노트북을 껐다 켜도 계속 재사용할 수 있다는 것이다.

In [8]:
%%time
# SQLite 캐시 파일에 이미 저장된 답이 있으므로 API 재호출 없이 바로 반환된다.
response = chain.invoke({"country":"한국"})
print(response.content)

대한민국은 동아시아 한반도 남부에 위치한 민주공화국으로, 수도는 서울이다. 오랜 역사와 전통문화를 바탕으로 한국전쟁 이후 빠른 산업화와 민주화를 이루었다. 세계적인 제조업·정보기술 강국이며, 반도체·자동차·조선·바이오 산업이 발달했다. 한류, 영화, 음악, 음식도 국제적으로 큰 영향력을 지닌다. 북한과 분단된 상황에서 안보와 평화, 저출생·고령화 같은 사회 문제에 대응하고 있다.
CPU times: total: 31.2 ms
Wall time: 6.11 ms


d:\moon0902\hanwha_0902\ex0917\.venv\Lib\site-packages\langchain_community\cache.py:265: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit list of allowed classes (or 'messages' for untrusted input that contains only chat messages) to suppress this warning.
  return [loads(row[0]) for row in rows]
